In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
import numpy as np
import pickle

In [ ]:
nx, ny, nz, nt = 512, 512, 100, 241
dx, dy, dz = 250.0, 250.0, 50.0 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
print('grid_vol=', grid_vol)

In [ ]:
ctl = 'goamazon_2pulse.largedom.r20251008.rerun'
ehe1 = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'

In [ ]:
def total_wq(casename):
    global nx, ny, nz, nt
    print(nx, ny, nz, nt)
    qfluxes = np.zeros((4, nz))
    areas = ['dom', 'tl', 'tul', 'tula']
    
    for i, area in enumerate(areas):
        filename = f'{casename}/pkl/qflux_{area}.pkl'
        try:
            with open(filename, 'rb') as f:
                qflux = pickle.load(f)
            # Denormalize and sum over domain
            qfluxes[i,:] = qflux
        except FileNotFoundError:
            print(f'Warning: {filename} not found')
            qfluxes[i,:] = np.nan
    
    return qfluxes

In [ ]:
# Load data for both cases
qflux_ctl = total_wq(ctl)
qflux_ehe1 = total_wq(ehe1)

In [ ]:
# Setup height coordinate
z = np.arange(0, nz) * dz  # meters
zi = (np.arange(0, nz+1) - 0.5) * dz  # interface heights

In [ ]:
# Plot comparison of CTL vs EHE1
pop_labels = ['domain', 'tracked large', 'tracked large full', 'tracked large full attached']
styles = ['--', 'None', '-', 'None']
markers = [None, 'o', None, '+']
markersizes = [None, 8, None, 8]
markerfacecolors = [None, 'none', None, 'auto']

fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))

for i in range(4):
# for i in [1,2,3]:
    ax.plot(qflux_ctl[i,:]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], markersize=markersizes[i], markerfacecolor=markerfacecolors[i], label=f'CTL - {pop_labels[i]}', color='black')
    ax.plot(qflux_ehe1[i,:]*1.0e3, z*1.0e-3, linestyle=styles[i], marker=markers[i], markersize=markersizes[i], markerfacecolor=markerfacecolors[i], label=f'EHE1 - {pop_labels[i]}', color='red')
ax.set_xlabel(r"$\overline{w'q_t'}$ ($10^{-3}$ g/kg m/s)")
ax.set_ylim((0, 5))
ax.set_ylabel('Height (km)')
plt.legend(loc='upper right', fontsize=14)
plt.show()

In [ ]:
# Calculate differences (EHE1 - CTL)
qflux_diff = qflux_ehe1 - qflux_ctl

# Create single figure
fig = plt.figure(figsize=(6, 9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))

for i in range(4):
    ax.plot(qflux_diff[i, :] * 1.0e3, z * 1.0e-3,
            linestyle=styles[i], 
            marker=markers[i],
            markersize=markersizes[i],
            markerfacecolor=markerfacecolors[i],
            color='black',
            linewidth=2, label=pop_labels[i])

ax.axvline(0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel(r"$\overline{w'q_t'}$ diff ($10^{-3}$ g/kg m/s)")
ax.set_ylabel('Height (km)')
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)
ax.legend(loc='best', fontsize=12)
plt.show()